# Handle Missing Values and Check Data Types

Input: `rhythm_filtered_cases.csv` (477 patients, the cohort after excluding Noise/Unclassifiable dominant rhythms).

Rules for missing values (decided per-column, based on the fraction of the 477 patients missing that value):
- **>50% missing** → drop the column entirely. With only 477 patients we don't have enough signal to use it reliably.
- **20–50% missing** → keep the column, but fill the gaps with the median of the non-missing values ("median imputation"). **Critically, before filling anything in, we create a companion `{column}_was_imputed` boolean column** so we can always tell which values are real measurements and which were filled in — this matters for the paper (so a reviewer can ask "how much of this feature is actually imputed?") and for any later sensitivity analysis.
- **<20% missing** → leave as-is (still `NaN` for the missing rows). XGBoost handles missing values natively, so there's no need to impute these, and imputing them would throw away the "this was unmeasured" signal for no benefit.
- **Categorical (text) columns** are never median-imputed (a median doesn't make sense for text). If any text column happens to fall in the 20–50% missing bucket, we leave it alone here and flag it for handling later when we encode categorical features.

Output: `imputed_477_cases.csv`.

In [ ]:
# pandas: dataframes. numpy: numeric helpers (median, NaN-aware functions). Path: file paths.
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..").resolve()
INPUT_CSV = BASE_DIR / "data" / "interim" / "rhythm_filtered_cases.csv"
OUTPUT_CSV = BASE_DIR / "data" / "interim" / "imputed_477_cases.csv"

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {INPUT_CSV}: {df.shape[0]} rows x {df.shape[1]} columns")

## Part 1: Missingness Report

In [ ]:
# For every column, count how many of the 477 rows are missing (NaN) and what % that is
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df) * 100)

missingness = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct.round(1),
    "dtype": df.dtypes,
}).sort_values("missing_pct", ascending=False)

pd.set_option("display.max_rows", None)
print(missingness[missingness["missing_count"] > 0].to_string())
print(f"\n{ (missingness['missing_count'] == 0).sum() } columns have zero missing values (not shown above).")

## Part 2: Drop Columns That Are >50% Missing

In [ ]:
DROP_THRESHOLD = 0.50   # drop if MORE than 50% missing

missing_frac = df.isnull().mean()   # same as missing_pct / 100, kept as a 0-1 fraction for comparisons

# Never drop the case ID, no matter what (it should be 0% missing anyway, but be explicit)
cols_to_drop = [c for c in df.columns if c != "caseid" and missing_frac[c] > DROP_THRESHOLD]

print(f"Dropping {len(cols_to_drop)} columns (>{DROP_THRESHOLD:.0%} missing):")
for c in cols_to_drop:
    print(f"  {c}: {missing_frac[c]:.1%} missing")

df = df.drop(columns=cols_to_drop)
print(f"\nShape after dropping: {df.shape}")

## Part 3: Median-Impute Numeric Columns That Are 20–50% Missing (With a "Was Imputed" Flag)

In [ ]:
IMPUTE_THRESHOLD = 0.20   # impute if missing fraction is > 20% and <= 50%

# Columns whose missing fraction falls in the 20-50% "impute" band
# (missing_frac was computed in Part 2, before any columns were dropped, but a column's
# own missing fraction doesn't change just because a DIFFERENT column got dropped)
impute_band_cols = [
    c for c in df.columns
    if c != "caseid" and IMPUTE_THRESHOLD < missing_frac[c] <= DROP_THRESHOLD
]

# Only numeric columns can be median-imputed — a median of text values is meaningless
numeric_impute_cols = [c for c in impute_band_cols if pd.api.types.is_numeric_dtype(df[c])]
non_numeric_impute_cols = [c for c in impute_band_cols if c not in numeric_impute_cols]

print(f"Numeric columns to median-impute (20-50% missing): {numeric_impute_cols}")
print(f"Non-numeric columns in the same missingness band (left alone, handle later when encoding): {non_numeric_impute_cols}")

In [ ]:
for col in numeric_impute_cols:
    # Step 1: record exactly which rows were missing BEFORE we touch the column.
    # This becomes a permanent, explicit record that survives even after the NaNs are filled in.
    flag_col = f"{col}_was_imputed"
    df[flag_col] = df[col].isna()

    # Step 2: compute the median using only the real (non-missing) values
    median_value = df[col].median()

    # Step 3: fill the missing values with that median
    df[col] = df[col].fillna(median_value)

    print(f"{col}: filled {df[flag_col].sum()} missing values with median {median_value:.3g}"
          f" (flag column '{flag_col}' added)")

print(f"\nShape after imputation: {df.shape}")

## Part 4: Columns Under 20% Missing — Left As-Is

In [ ]:
# These still contain real NaN values on purpose. XGBoost natively handles missing
# values during training, so there's no need to fill these in, and doing so would
# only throw away information for no benefit.
low_missing_cols = [
    c for c in df.columns
    if c != "caseid" and not c.endswith("_was_imputed")
    and 0 < missing_frac.get(c, 0) <= IMPUTE_THRESHOLD
]
print(f"{len(low_missing_cols)} columns left as-is (<= {IMPUTE_THRESHOLD:.0%} missing, NaN preserved):")
for c in low_missing_cols:
    print(f"  {c}: {missing_frac[c]:.1%} missing")

## Part 5: Check and Fix Data Types

In [ ]:
print("Current column data types:")
print(df.dtypes.to_string())

In [ ]:
# Automatically detect any column that LOOKS like text (object dtype) but is actually
# fully numeric underneath (e.g. '72.0' stored as a string instead of the number 72.0).
# For each object column, try converting every value to a number; if every non-missing
# value converts successfully, the column was mistakenly stored as text and we fix it.
object_cols = df.select_dtypes(include="object").columns.tolist()
fixed_cols = []

for col in object_cols:
    converted = pd.to_numeric(df[col], errors="coerce")
    originally_present = df[col].notna()
    successfully_converted = converted.notna()
    if originally_present.sum() > 0 and (originally_present == successfully_converted).all():
        df[col] = converted
        fixed_cols.append(col)

print(f"Columns converted from text to numeric: {fixed_cols if fixed_cols else 'none — all text columns are genuinely categorical'}")

In [ ]:
# Spot-check the columns the project guide specifically asks us to verify
print("caseid dtype:", df["caseid"].dtype, "(should be integer)")
print("age dtype:", df["age"].dtype, "(should be numeric)")
print("weight dtype:", df["weight"].dtype, "(should be numeric)")
print("height dtype:", df["height"].dtype, "(should be numeric)")
print("dominant_rhythm dtype:", df["dominant_rhythm"].dtype, "(should be object/string)")

## Part 6: Save the Cleaned Dataframe

In [ ]:
assert df["caseid"].is_unique, "caseid should still be unique"

remaining_missing = df.isnull().sum()
print("Columns that STILL have missing values after this notebook (expected: only the <20%-missing ones from Part 4):")
print(remaining_missing[remaining_missing > 0].sort_values(ascending=False).to_string())

df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {df.shape[0]} rows x {df.shape[1]} columns to {OUTPUT_CSV.resolve()}")